<a href="https://colab.research.google.com/github/BigingiIan/ml_models/blob/main/E_143496.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


**Overview**

There are three cells:

* In the first cell, **enter your 6-digit student number**. This will be used to randomize a dataset unique to you.
* Run the second and third cell to downlod and prepare a dataset from Hugging Face

---

**Primary Task**
- Using the knowledge you've acquired in the unit ICS 4104 Machine Learning,
implement a simple ANN pipeline to classify text(s).
- **Use at most 70% percent** of your data for training. You can decide the percentages/ratios for any other splits you may need.
- [REFER TO THIS DOCUMENT FOR MORE DETAILS](https://docs.google.com/document/d/1vpDA4kL-Kzf_s0XdzrlUkVjE0DbRa4kRrfAFiHZ3bD4/edit?usp=sharing)

---

**Guidelines:**

* START FROM THE CELL MARKED **CELL 4**
* You are limited to using only the imported libraries. DO NOT ADD ANY IMPORTS.
* You have 45 minutes to complete the task and submit your work [here](https://forms.gle/bFmW2BqZeB71NpAH8). OR 'https://forms.gle/bFmW2BqZeB71NpAH8'


**AOB:**
- Make a copy of this file and rename it with your student number and group e.g E-111222 or D-999888
- I'll be available on email. Incase anyone has any issues,** mail me direclty, no need to go through the class reps.**


In [3]:
# CELL 1, assign your student number to the valirable below, e.g. student_id = 111222 or student_id = 123456 or student_id = 987654

student_id = 143496

In [4]:
# CELL 2:

# DO NOT MODIFY THIS CELL

import random
import pandas as pd
import numpy as np

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, GRU, RNN

import warnings
warnings.filterwarnings('ignore')

# DO NOT MODIFY THIS CELL

In [5]:
# CELL 3:

    # Loads a dataset based on your student number and presents your with a two column dataframe
    # Dataframe has an input column 'X' and an output column 'Y'

# DO NOT MODIFY THIS CELL

ds = load_dataset("Tobi-Bueck/customer-support-tickets")
df = ds["train"].to_pandas()
df = df[df['language'] == 'en']
df = df[['body', 'type']]
df = df.rename(columns={'body': 'X', 'type': 'Y'})
df = df.sample(n=2000 + (student_id % 1000), random_state=student_id)
df = df.reset_index(drop=True)

print(f"Your ({student_id}) dataset has: {df.shape[0]} rows (examples) and {df.shape[1]} columns (features)")
print("---" * 20)

df.head()

# DO NOT MODIFY THIS CELL

Your (143496) dataset has: 2496 rows (examples) and 2 columns (features)
------------------------------------------------------------


,X,Y
0,There is a potential data breach that has comp...,Problem
1,"Customer Support, I am currently working on a ...",Incident
2,"Dear Customer Support, I am contacting you to ...",Incident
3,There was a billing discrepancy while attempti...,Incident
4,"Dear Customer Support,\n\nI am writing to repo...",Problem


----------

## START YOUR SOLUTION(s) IN THE CELLS BELLOW

In [6]:
# CELL 4:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Y'] = le.fit_transform(df['Y'])
df.head()




,X,Y
0,There is a potential data breach that has comp...,2
1,"Customer Support, I am currently working on a ...",1
2,"Dear Customer Support, I am contacting you to ...",1
3,There was a billing discrepancy while attempti...,1
4,"Dear Customer Support,\n\nI am writing to repo...",2


In [7]:
# CELL 5:
print(dict(zip(le.classes_, le.transform(le.classes_))))




{'Change': np.int64(0), 'Incident': np.int64(1), 'Problem': np.int64(2), 'Request': np.int64(3)}


In [8]:
# CELL 6:
df = df.dropna()





In [10]:
# CELL 7:
tr_in, te_in, tr_out, te_out = train_test_split(df['X'], df['Y'], test_size=0.3, random_state=student_id)

tr_in, val_in, tr_out, val_out = train_test_split(tr_in, tr_out, test_size=0.5, random_state=student_id)

print(tr_in.shape, val_in.shape, te_in.shape)

(873,) (874,) (749,)


In [11]:
# CELL 8:
tokenizer = Tokenizer(oov_token='<OOV>')
tokenizer.fit_on_texts(tr_in)

tr_sq = tokenizer.texts_to_sequences(tr_in)
val_sq = tokenizer.texts_to_sequences(val_in)
te_sq = tokenizer.texts_to_sequences(te_in)

max_len = max(
    max(map(len, tr_sq)),
    max(map(len, val_sq)),
    max(map(len, te_sq))
)

print(max_len)

tr_sq = pad_sequences(tr_sq, maxlen=max_len)
val_sq = pad_sequences(val_sq, maxlen=max_len)
te_sq = pad_sequences(te_sq, maxlen=max_len)

print(tr_sq.shape, val_sq.shape, te_sq.shape)

213
(873, 213) (874, 213) (749, 213)


In [15]:
# CELL 9:
vocab_size = len(tokenizer.word_index) + 1
num_classes = len(le.classes_)

model = Sequential()

model.add(Embedding(vocab_size, 128, input_length=max_len))

model.add(LSTM(64))

model.add(Dropout(0.2))

model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))

model.add(Dense(num_classes, activation='softmax'))

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [16]:
# CELL 10:
history = model.fit(
  tr_sq, tr_out,
  validation_data=(val_sq, val_out),
  epochs=25,
  batch_size=32)


Epoch 1/25
28/28 ━━━━━━━━━━━━━━━━━━━━ 8s 177ms/step - accuracy: 0.3322 - loss: 1.3421 - val_accuracy: 0.5904 - val_loss: 1.2396
Epoch 2/25
28/28 ━━━━━━━━━━━━━━━━━━━━ 7s 232ms/step - accuracy: 0.6002 - loss: 1.1069 - val_accuracy: 0.6419 - val_loss: 0.9121
Epoch 3/25
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 166ms/step - accuracy: 0.6369 - loss: 0.8339 - val_accuracy: 0.6808 - val_loss: 0.8736
Epoch 4/25
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 194ms/step - accuracy: 0.6712 - loss: 0.7217 - val_accuracy: 0.6888 - val_loss: 0.6860
Epoch 5/25
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 192ms/step - accuracy: 0.6884 - loss: 0.6166 - val_accuracy: 0.7014 - val_loss: 0.6492
Epoch 6/25
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 165ms/step - accuracy: 0.7434 - loss: 0.5419 - val_accuracy: 0.7334 - val_loss: 0.6358
Epoch 7/25
28/28 ━━━━━━━━━━━━━━━━━━━━ 7s 230ms/step - accuracy: 0.8144 - loss: 0.4705 - val_accuracy: 0.7368 - val_loss: 0.6141
Epoch 8/25
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 180ms/step - accuracy: 0.8293 - loss: 0.4278 - val_accuracy: 0.

In [17]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 213, 128)       │       334,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 4)              │            68 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,158,878 (4.42 MB)

 Trainable params: 386,292 (1.47 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 772,586 (2.95 MB)

In [18]:
loss, accuracy = model.evaluate(te_sq, te_out)
print(f"Test Loss: {loss}, Test Accuracy: {accuracy}")

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.7116 - loss: 1.4941
Test Loss: 1.4940621852874756, Test Accuracy: 0.7116155028343201


In [20]:
model.save('143496.keras')